In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()


gemini_api_key = os.getenv("GEMINI_API_KEY")
print(gemini_api_key)

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key

In [ ]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

In [ ]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="hi, i am pratham ai engineer")])

In [ ]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="hi, i am pratham and i am ai engineer"),
        AIMessage(content="Hello Pratham, nice to meet you! As an AI engineer, you must be working on some exciting projects"),
        HumanMessage(content="what is my name and what de i do?")
    ]
)

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

##how one chat session is different from other session
store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [ ]:
config = {"configurable": {"session_id": "chat1"}}

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="i am pratham and i am ai engineer")],
    config=config
)

In [ ]:
with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config=config
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt=ChatPromptTemplate.from_messages(
    [
        ("system","answer q as per best of your capacity"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain=prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="My name is Pratham")]})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [ ]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="i am pratham")],
    config=config
)

response

In [ ]:
## prompt templates

from langchain_core.prompts import ChatPromptTemplate

generic_temp="Translate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_temp),
     MessagesPlaceholder(variable_name="messages")]
)

chain=prompt|model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="my anme is pratham")],"language":"Hindi"})

In [ ]:
response.content

In [ ]:
###manging conversation history

In [ ]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"

)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

In [ ]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

response=chain.invoke(
    {
    "messages":messages+[HumanMessage(content="What icecream do I like")],
    "language":"English"
})
response.content